In [1]:
import math
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

In [2]:
# Dataset 4 dokumen pendek, masing-masing 1-3 kalimat
documents = [
    "Sistem komputer mengolah data menggunakan perangkat keras dan perangkat lunak.",
    "Jaringan komputer menghubungkan perangkat untuk bertukar data secara cepat dan aman.",
    "Kecerdasan buatan menggunakan data untuk mempelajari pola dan menghasilkan prediksi.",
    "Sistem temu kembali mencari dokumen berdasarkan kemiripan data dan kebutuhan pengguna."
]

doc_names = [f"Dokumen {i}" for i in range(1, 5)]

# Preprocessing sederhana: case folding + tokenisasi + stopwords removal
# Daftar stopword dibuat eksplisit agar perhitungan manual dan sklearn memakai vocabulary yang sama.
stopwords = {
    "dan", "yang", "untuk", "dengan", "secara", "dari", "ke", "di",
    "pada", "dalam", "berdasarkan", "serta", "atau", "akan", "dapat",
    "menggunakan"
}

def simple_preprocess(text):
    text = text.lower()
    tokens = text.replace(".", "").replace(",", "").split()
    tokens = [token for token in tokens if token not in stopwords]
    return tokens

tokenized_docs = [simple_preprocess(doc) for doc in documents]
processed_docs = [" ".join(tokens) for tokens in tokenized_docs]

pd.DataFrame({
    "Dokumen": doc_names,
    "Teks Asli": documents,
    "Hasil Preprocessing": processed_docs
})

,Dokumen,Teks Asli,Hasil Preprocessing
0,Dokumen 1,Sistem komputer mengolah data menggunakan pera...,sistem komputer mengolah data perangkat keras ...
1,Dokumen 2,Jaringan komputer menghubungkan perangkat untu...,jaringan komputer menghubungkan perangkat bert...
2,Dokumen 3,Kecerdasan buatan menggunakan data untuk mempe...,kecerdasan buatan data mempelajari pola mengha...
3,Dokumen 4,Sistem temu kembali mencari dokumen berdasarka...,sistem temu kembali mencari dokumen kemiripan ...


In [3]:
# Vocabulary seluruh dokumen
vocabulary = sorted(set(term for tokens in tokenized_docs for term in tokens))

bow = pd.DataFrame(0, index=doc_names, columns=vocabulary)

for doc_name, tokens in zip(doc_names, tokenized_docs):
    counts = Counter(tokens)
    for term, count in counts.items():
        bow.loc[doc_name, term] = count

print("Vocabulary:")
print(vocabulary)
print("\nMatriks Bag-of-Words (raw count):")
bow

Vocabulary:
['aman', 'bertukar', 'buatan', 'cepat', 'data', 'dokumen', 'jaringan', 'kebutuhan', 'kecerdasan', 'kembali', 'kemiripan', 'keras', 'komputer', 'lunak', 'mempelajari', 'mencari', 'menghasilkan', 'menghubungkan', 'mengolah', 'pengguna', 'perangkat', 'pola', 'prediksi', 'sistem', 'temu']

Matriks Bag-of-Words (raw count):


,aman,bertukar,buatan,cepat,data,dokumen,jaringan,kebutuhan,kecerdasan,kembali,kemiripan,keras,komputer,lunak,mempelajari,mencari,menghasilkan,menghubungkan,mengolah,pengguna,perangkat,pola,prediksi,sistem,temu
Dokumen 1,0,0,0,0,1,0,0,0,0,0,0,1,1,1,0,0,0,0,1,0,2,0,0,1,0
Dokumen 2,1,1,0,1,1,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0
Dokumen 3,0,0,1,0,1,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,1,1,0,0
Dokumen 4,0,0,0,0,1,1,0,1,0,1,1,0,0,0,0,1,0,0,0,1,0,0,0,1,1


2. Term Frequency (TF)

Rumus yang digunakan:

TF(term, dokumen) = jumlah kemunculan term / jumlah seluruh token pada dokumen

Dengan demikian, TF menunjukkan proporsi sebuah term di dalam dokumen.

In [4]:
doc_lengths = {doc: len(tokens) for doc, tokens in zip(doc_names, tokenized_docs)}

tf = bow.copy().astype(float)
for doc in doc_names:
    tf.loc[doc] = tf.loc[doc] / doc_lengths[doc]

print("Jumlah token tiap dokumen:", doc_lengths)
print("\nMatriks TF:")
tf.round(4)

Jumlah token tiap dokumen: {'Dokumen 1': 8, 'Dokumen 2': 8, 'Dokumen 3': 7, 'Dokumen 4': 9}

Matriks TF:


,aman,bertukar,buatan,cepat,data,dokumen,jaringan,kebutuhan,kecerdasan,kembali,kemiripan,keras,komputer,lunak,mempelajari,mencari,menghasilkan,menghubungkan,mengolah,pengguna,perangkat,pola,prediksi,sistem,temu
Dokumen 1,0.000,0.000,0.0000,0.000,0.1250,0.0000,0.000,0.0000,0.0000,0.0000,0.0000,0.125,0.125,0.125,0.0000,0.0000,0.0000,0.000,0.125,0.0000,0.250,0.0000,0.0000,0.1250,0.0000
Dokumen 2,0.125,0.125,0.0000,0.125,0.1250,0.0000,0.125,0.0000,0.0000,0.0000,0.0000,0.000,0.125,0.000,0.0000,0.0000,0.0000,0.125,0.000,0.0000,0.125,0.0000,0.0000,0.0000,0.0000
Dokumen 3,0.000,0.000,0.1429,0.000,0.1429,0.0000,0.000,0.0000,0.1429,0.0000,0.0000,0.000,0.000,0.000,0.1429,0.0000,0.1429,0.000,0.000,0.0000,0.000,0.1429,0.1429,0.0000,0.0000
Dokumen 4,0.000,0.000,0.0000,0.000,0.1111,0.1111,0.000,0.1111,0.0000,0.1111,0.1111,0.000,0.000,0.000,0.0000,0.1111,0.0000,0.000,0.000,0.1111,0.000,0.0000,0.0000,0.1111,0.1111


3. Document Frequency (DF) dan Inverse Document Frequency (IDF)

Rumus yang digunakan secara manual:

DF(term)= jumlah dokumen yang mengandung term.
IDF(term) = log10(N / DF(term)), dengan N = jumlah dokumen.

Rumus ini dipilih agar perhitungan manual jelas dan konsisten.

In [5]:
N = len(documents)

df_values = (bow > 0).sum(axis=0)
idf_values = pd.Series({
    term: math.log10(N / df_values[term])
    for term in vocabulary
})

df_idf = pd.DataFrame({
    "Term": vocabulary,
    "DF": [df_values[t] for t in vocabulary],
    "IDF": [idf_values[t] for t in vocabulary]
})

df_idf

,Term,DF,IDF
0,aman,1,0.60206
1,bertukar,1,0.60206
2,buatan,1,0.60206
3,cepat,1,0.60206
4,data,4,0.00000
5,dokumen,1,0.60206
6,jaringan,1,0.60206
7,kebutuhan,1,0.60206
8,kecerdasan,1,0.60206
9,kembali,1,0.60206


4. TF-IDF Manual

Rumus:

TF-IDF(term, dokumen) = TF(term, dokumen) × IDF(term)

Karena term yang muncul pada lebih sedikit dokumen memiliki IDF lebih tinggi, term yang khas pada suatu dokumen cenderung mendapatkan bobot TF-IDF yang lebih besar.

In [6]:
tfidf_manual = tf.copy()
for term in vocabulary:
    tfidf_manual[term] = tf[term] * idf_values[term]

print("Matriks TF-IDF manual:")
tfidf_manual.round(4)

Matriks TF-IDF manual:


,aman,bertukar,buatan,cepat,data,dokumen,jaringan,kebutuhan,kecerdasan,kembali,kemiripan,keras,komputer,lunak,mempelajari,mencari,menghasilkan,menghubungkan,mengolah,pengguna,perangkat,pola,prediksi,sistem,temu
Dokumen 1,0.0000,0.0000,0.000,0.0000,0.0,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.0753,0.0376,0.0753,0.000,0.0000,0.000,0.0000,0.0753,0.0000,0.0753,0.000,0.000,0.0376,0.0000
Dokumen 2,0.0753,0.0753,0.000,0.0753,0.0,0.0000,0.0753,0.0000,0.000,0.0000,0.0000,0.0000,0.0376,0.0000,0.000,0.0000,0.000,0.0753,0.0000,0.0000,0.0376,0.000,0.000,0.0000,0.0000
Dokumen 3,0.0000,0.0000,0.086,0.0000,0.0,0.0000,0.0000,0.0000,0.086,0.0000,0.0000,0.0000,0.0000,0.0000,0.086,0.0000,0.086,0.0000,0.0000,0.0000,0.0000,0.086,0.086,0.0000,0.0000
Dokumen 4,0.0000,0.0000,0.000,0.0000,0.0,0.0669,0.0000,0.0669,0.000,0.0669,0.0669,0.0000,0.0000,0.0000,0.000,0.0669,0.000,0.0000,0.0000,0.0669,0.0000,0.000,0.000,0.0334,0.0669


5. TF-IDF dengan scikit-learn

TfidfVectorizer secara default menggunakan IDF yang dihaluskan (*smooth IDF*) dan melakukan normalisasi L2 pada vektor dokumen. Karena itu, nilainya tidak harus sama persis dengan perhitungan manual di atas. Perbandingan tetap dapat dilakukan untuk melihat kesesuaian pola pembobotannya.

In [7]:
# Gunakan vocabulary yang sama dan preprocessing yang sama.
vectorizer = TfidfVectorizer(
    vocabulary=vocabulary,
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None
)

X_sklearn = vectorizer.fit_transform(processed_docs)

tfidf_sklearn = pd.DataFrame(
    X_sklearn.toarray(),
    index=doc_names,
    columns=vectorizer.get_feature_names_out()
)

print("Matriks TF-IDF dari scikit-learn:")
tfidf_sklearn.round(4)

Matriks TF-IDF dari scikit-learn:


,aman,bertukar,buatan,cepat,data,dokumen,jaringan,kebutuhan,kecerdasan,kembali,kemiripan,keras,komputer,lunak,mempelajari,mencari,menghasilkan,menghubungkan,mengolah,pengguna,perangkat,pola,prediksi,sistem,temu
Dokumen 1,0.0000,0.0000,0.0000,0.0000,0.1972,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3779,0.2980,0.3779,0.0000,0.0000,0.0000,0.0000,0.3779,0.0000,0.5959,0.0000,0.0000,0.2980,0.0000
Dokumen 2,0.3918,0.3918,0.0000,0.3918,0.2044,0.0000,0.3918,0.0000,0.0000,0.0000,0.0000,0.0000,0.3089,0.0000,0.0000,0.0000,0.0000,0.3918,0.0000,0.0000,0.3089,0.0000,0.0000,0.0000,0.0000
Dokumen 3,0.0000,0.0000,0.3993,0.0000,0.2084,0.0000,0.0000,0.0000,0.3993,0.0000,0.0000,0.0000,0.0000,0.0000,0.3993,0.0000,0.3993,0.0000,0.0000,0.0000,0.0000,0.3993,0.3993,0.0000,0.0000
Dokumen 4,0.0000,0.0000,0.0000,0.0000,0.1857,0.3559,0.0000,0.3559,0.0000,0.3559,0.3559,0.0000,0.0000,0.0000,0.0000,0.3559,0.0000,0.0000,0.0000,0.3559,0.0000,0.0000,0.0000,0.2806,0.3559


Catatan perbandingan

Perhitungan manual menggunakan IDF = log10(N/DF) tanpa normalisasi vektor, sedangkan TfidfVectorizer menggunakan formulasi IDF terhalus dan normalisasi L2 secara default. Oleh karena itu, angka manual dan scikit-learn dapat berbeda, tetapi term yang relatif penting dalam setiap dokumen dapat dibandingkan.

In [8]:
# Term dengan bobot TF-IDF manual tertinggi pada setiap dokumen
rows = []

for doc in doc_names:
    term = tfidf_manual.loc[doc].idxmax()
    value = tfidf_manual.loc[doc, term]
    rows.append({
        "Dokumen": doc,
        "Term TF-IDF Tertinggi (Manual)": term,
        "Bobot": round(value, 4)
    })

highest_manual = pd.DataFrame(rows)
highest_manual

,Dokumen,Term TF-IDF Tertinggi (Manual),Bobot
0,Dokumen 1,keras,0.0753
1,Dokumen 2,aman,0.0753
2,Dokumen 3,buatan,0.0860
3,Dokumen 4,dokumen,0.0669


In [9]:
# Term dengan bobot TF-IDF scikit-learn tertinggi pada setiap dokumen
rows = []

for doc in doc_names:
    term = tfidf_sklearn.loc[doc].idxmax()
    value = tfidf_sklearn.loc[doc, term]
    rows.append({
        "Dokumen": doc,
        "Term TF-IDF Tertinggi (Sklearn)": term,
        "Bobot": round(value, 4)
    })

highest_sklearn = pd.DataFrame(rows)
highest_sklearn

,Dokumen,Term TF-IDF Tertinggi (Sklearn),Bobot
0,Dokumen 1,perangkat,0.5959
1,Dokumen 2,aman,0.3918
2,Dokumen 3,buatan,0.3993
3,Dokumen 4,dokumen,0.3559


Hasil Analisis

Term dengan bobot TF-IDF tertinggi pada masing-masing dokumen cenderung merupakan term yang cukup sering muncul pada dokumen tersebut tetapi tidak tersebar pada banyak dokumen lain. Hal ini membuat term tersebut lebih mampu membedakan satu dokumen dari dokumen lainnya. Sebaliknya, term yang muncul pada banyak dokumen memperoleh nilai IDF lebih rendah sehingga kontribusinya terhadap pembeda antar dokumen juga lebih kecil. Perbedaan angka antara perhitungan manual dan scikit-learn terjadi karena formulasi IDF dan normalisasi yang digunakan TfidfVectorizer berbeda dari rumus manual yang digunakan pada tugas ini.